# MODUL PRAKTIKUM BIG DATA
## Pertemuan 14 — Unsupervised Learning: Segmentasi Pelanggan dengan K-Means Clustering

| | |
|---|---|
| **Mata Kuliah** | Praktikum Big Data |
| **Program Studi** | Teknologi Informasi — Universitas Tidar |
| **Pertemuan** | 14 |
| **Topik** | Unsupervised Learning, Feature Scaling, K-Means Clustering, Elbow Method |
| **Estimasi Waktu** | 150 menit |
| **Prasyarat** | Modul Pertemuan 1-13 selesai (memahami Spark MLlib: `Pipeline`, regresi, klasifikasi, model persistence) |

---

### Cara Menggunakan Modul Ini

**Tidak ada instalasi baru.** `KMeans` dan `StandardScaler` adalah bagian bawaan `pyspark.ml`, modul yang sama yang telah dipakai sejak Pertemuan 11. Seluruh modul dikerjakan di Jupyter Notebook.

> **Konsistensi versi:** Tetap **Spark/PySpark 3.5.9**, **Python 3.11** — tidak ada perubahan versi apa pun.


---
## Recap Pertemuan Sebelumnya

- [ ] Memahami `Pipeline`, `StringIndexer`, `VectorAssembler` dari Pertemuan 11-12
- [ ] Memahami cara melatih & mengevaluasi model regresi (Pertemuan 11) dan klasifikasi (Pertemuan 12)
- [ ] Memahami model persistence (`save`/`load`) dari Pertemuan 13

## Tujuan Pembelajaran

Setelah menyelesaikan Pertemuan 14, mahasiswa mampu:
1. Menjelaskan perbedaan mendasar antara *supervised* dan *unsupervised learning*.
2. Menjelaskan mengapa **penyekalaan fitur** (*feature scaling*) penting untuk algoritma berbasis jarak seperti K-Means.
3. Menentukan jumlah klaster optimal menggunakan **Elbow Method**.
4. Melatih model K-Means dan menginterpretasikan hasil klaster untuk kebutuhan bisnis.

---

## 14.1 Supervised vs Unsupervised Learning

Pada Pertemuan 11 (regresi) dan Pertemuan 12 (klasifikasi), setiap data pelatihan kita **selalu memiliki label** — nilai `pendapatan` yang sesungguhnya, atau `label` 0/1 yang sudah diketahui. Model belajar memetakan fitur ke label yang **sudah ada jawabannya**. Ini disebut ***supervised learning*** (pembelajaran terarah/berlabel).

**Unsupervised learning** adalah kebalikannya: **tidak ada label sama sekali**. Model harus menemukan **pola atau struktur tersembunyi** dalam data dengan sendirinya. Salah satu tugas unsupervised learning yang paling umum adalah ***clustering*** (pengelompokan) — mengelompokkan data yang mirip satu sama lain ke dalam kelompok (klaster) yang sama, tanpa diberi tahu terlebih dahulu kelompok apa saja yang ada.

| Aspek | Supervised (Pertemuan 11-12) | Unsupervised (Pertemuan 14) |
|---|---|---|
| **Label pada data training** | Ada (`labelCol` wajib diisi) | Tidak ada |
| **Tujuan model** | Memprediksi label yang benar | Menemukan struktur/pola tersembunyi |
| **Contoh algoritma** | `LinearRegression`, `LogisticRegression` | `KMeans` |
| **Contoh kasus bisnis** | "Akankah pelanggan ini membatalkan pesanan?" | "Kelompok pelanggan seperti apa saja yang kita miliki?" |

**Studi kasus hari ini:** Platform e-commerce (skenario yang menemani kita sejak Pertemuan 2) ingin memahami **segmen-segmen pelanggan** yang ada berdasarkan pola belanja mereka — tanpa menentukan sebelumnya "pelanggan tipe apa saja yang harus dicari". Inilah kasus klasik yang paling cocok diselesaikan dengan *clustering*.


---
## 14.2 Persiapan: SparkSession & Dataset Perilaku Pelanggan

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = SparkSession.builder \
    .appName("Pertemuan14-KMeansClustering") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

print("SparkSession siap. Versi Spark:", spark.version)

In [ ]:
import numpy as np
import pandas as pd

np.random.seed(14)
n_per_kelompok = 200

# Data disimulasikan mengandung 3 pola perilaku pelanggan yang berbeda secara alami
# (pada data dunia nyata, kita TIDAK akan tahu pola ini sebelumnya -- itulah tujuan clustering!)
kelompok_1 = pd.DataFrame({  # sering belanja, nilai kecil per transaksi
    "frekuensi_transaksi": np.random.normal(20, 3, n_per_kelompok),
    "rata_rata_nilai": np.random.normal(50000, 10000, n_per_kelompok),
})
kelompok_2 = pd.DataFrame({  # jarang belanja, nilai besar per transaksi
    "frekuensi_transaksi": np.random.normal(3, 1, n_per_kelompok),
    "rata_rata_nilai": np.random.normal(500000, 50000, n_per_kelompok),
})
kelompok_3 = pd.DataFrame({  # frekuensi & nilai moderat
    "frekuensi_transaksi": np.random.normal(10, 2, n_per_kelompok),
    "rata_rata_nilai": np.random.normal(150000, 20000, n_per_kelompok),
})

data = pd.concat([kelompok_1, kelompok_2, kelompok_3], ignore_index=True)
data["frekuensi_transaksi"] = data["frekuensi_transaksi"].clip(lower=1)
data["rata_rata_nilai"] = data["rata_rata_nilai"].clip(lower=10000)
data["pelanggan_id"] = [f"CUST-{i:04d}" for i in range(len(data))]

df = spark.createDataFrame(data)
df.show(5)
print("Total pelanggan:", df.count())

Dataset ini berisi dua atribut perilaku tiap pelanggan: **`frekuensi_transaksi`** (berapa kali bertransaksi dalam sebulan) dan **`rata_rata_nilai`** (rata-rata nilai rupiah tiap transaksi). Perhatikan **tidak ada kolom label/kategori apa pun** — inilah ciri khas data unsupervised learning.


---
## 14.3 Mengapa Feature Scaling Penting untuk K-Means?

K-Means bekerja dengan menghitung **jarak** antar titik data (secara default, jarak Euclidean) untuk menentukan titik mana yang "mirip" dan layak dikelompokkan bersama. Perhatikan skala kedua fitur kita sangat berbeda: `frekuensi_transaksi` berkisar 1-25, sementara `rata_rata_nilai` berkisar puluhan ribu hingga setengah juta rupiah.

**Tanpa penyekalaan**, fitur `rata_rata_nilai` akan **mendominasi** perhitungan jarak hanya karena angkanya jauh lebih besar secara nominal — bukan karena benar-benar lebih penting secara bisnis. Model bisa saja hampir sepenuhnya mengelompokkan berdasarkan `rata_rata_nilai` saja dan mengabaikan `frekuensi_transaksi`.

> **Mengapa ini tidak menjadi masalah di Pertemuan 11-12?** Algoritma seperti `LinearRegression`, `LogisticRegression`, dan terutama model berbasis tree (`RandomForestClassifier`) tidak menghitung jarak antar titik data secara langsung, sehingga jauh lebih toleran terhadap skala fitur yang berbeda-beda. **K-Means (dan hampir seluruh algoritma berbasis jarak lainnya) jauh lebih sensitif** — feature scaling menjadi wajib, bukan opsional.

**`StandardScaler`** mengubah setiap fitur agar memiliki rata-rata 0 dan simpangan baku (standar deviasi) 1 — sehingga seluruh fitur berkontribusi secara setara dalam perhitungan jarak.

In [ ]:
from pyspark.ml.feature import VectorAssembler, StandardScaler

# Tahap 1: gabungkan fitur mentah menjadi satu vector (pola yang sudah familiar sejak Pertemuan 11)
assembler = VectorAssembler(
    inputCols=["frekuensi_transaksi", "rata_rata_nilai"],
    outputCol="features_mentah"
)
df_assembled = assembler.transform(df)

# Tahap 2: StandardScaler -- withMean & withStd memastikan hasil benar-benar mean=0, std=1
scaler = StandardScaler(inputCol="features_mentah", outputCol="features", withStd=True, withMean=True)
scaler_model = scaler.fit(df_assembled)
df_scaled = scaler_model.transform(df_assembled)

df_scaled.select("frekuensi_transaksi", "rata_rata_nilai", "features_mentah", "features").show(5, truncate=False)

Perhatikan kolom `features` (hasil scaling) sekarang berisi angka-angka kecil di sekitar 0 (bisa positif atau negatif) untuk **kedua** fitur — tidak ada lagi satu fitur yang secara nominal jauh lebih besar dari fitur lainnya.


---
## 14.4 Menentukan Jumlah Klaster (k) dengan Elbow Method

Berbeda dengan klasifikasi (di mana jumlah kelas biasanya sudah jelas, mis. 0/1), **K-Means mengharuskan kita menentukan jumlah klaster (k) di awal** — namun pada data dunia nyata, kita sering tidak tahu berapa jumlah kelompok "alami" yang sebenarnya ada.

**Elbow Method** adalah teknik umum untuk membantu menentukan k: kita mencoba melatih model dengan beberapa nilai k berbeda, lalu mengamati metrik **WSSSE** (*Within Set Sum of Squared Errors* — total jarak kuadrat tiap titik ke pusat klasternya). Semakin besar k, WSSSE pasti akan terus menurun (klaster makin banyak, makin "pas") — namun kita mencari titik di mana **penurunannya mulai melandai drastis**, membentuk seperti siku (*elbow*) pada grafik.

In [ ]:
from pyspark.ml.clustering import KMeans

hasil_wssse = []
for k in range(2, 8):
    kmeans_coba = KMeans(featuresCol="features", k=k, seed=42)
    model_coba = kmeans_coba.fit(df_scaled)
    wssse = model_coba.summary.trainingCost
    hasil_wssse.append((k, wssse))
    print(f"k={k}  ->  WSSSE = {wssse:.2f}")

In [ ]:
import matplotlib.pyplot as plt

k_list = [item[0] for item in hasil_wssse]
wssse_list = [item[1] for item in hasil_wssse]

plt.figure(figsize=(8, 5))
plt.plot(k_list, wssse_list, marker="o", color="#1F3864")
plt.title("Elbow Method: Mencari Jumlah Klaster Optimal")
plt.xlabel("Jumlah Klaster (k)")
plt.ylabel("WSSSE")
plt.xticks(k_list)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

Perhatikan grafik: penurunan WSSSE dari k=2 ke k=3 sangat tajam, namun dari k=3 dan seterusnya penurunannya jauh lebih landai. Titik "siku" ini berada di **k=3** — mengindikasikan 3 klaster adalah jumlah yang paling masuk akal untuk data kita (cocok dengan desain data simulasi kita yang memang memiliki 3 pola alami, meski dalam praktiknya kita tidak akan tahu ini sebelumnya).

> **Elbow Method bersifat panduan, bukan aturan mutlak.** Pada data dunia nyata, "siku"-nya sering tidak setegas ini — penentuan k akhir tetap membutuhkan pertimbangan konteks bisnis (mis. tim marketing hanya sanggup menjalankan maksimal 4 kampanye berbeda, sehingga k=4 dipilih meski k=5 sedikit lebih optimal secara matematis).


---
## 14.5 Melatih Model K-Means Final

In [ ]:
kmeans = KMeans(featuresCol="features", k=3, seed=42)
model = kmeans.fit(df_scaled)

predictions = model.transform(df_scaled)
predictions.select("pelanggan_id", "frekuensi_transaksi", "rata_rata_nilai", "prediction").show(10)

print("\nJumlah pelanggan per klaster:")
predictions.groupBy("prediction").count().orderBy("prediction").show()

### Evaluasi: Silhouette Score

Karena tidak ada label "benar" untuk dibandingkan (ingat, ini unsupervised!), metrik evaluasi klasifikasi seperti accuracy/F1 pada Pertemuan 12 **tidak berlaku** di sini. **Silhouette Score** mengukur seberapa baik klaster terbentuk secara internal — seberapa dekat suatu titik dengan klasternya sendiri, dibandingkan jarak ke klaster terdekat lainnya. Skalanya -1 hingga 1, semakin **mendekati 1** semakin baik (klaster terpisah jelas satu sama lain).

In [ ]:
from pyspark.ml.evaluation import ClusteringEvaluator

evaluator = ClusteringEvaluator(featuresCol="features")
silhouette = evaluator.evaluate(predictions)

print(f"Silhouette Score: {silhouette:.4f}")

### Menginterpretasikan Klaster Secara Bisnis

Angka klaster (0, 1, 2) sendiri **tidak memiliki makna** — tugas kita adalah menerjemahkannya ke dalam bahasa bisnis dengan melihat karakteristik rata-rata tiap klaster **pada skala aslinya** (bukan hasil scaling, yang sudah tidak intuitif dibaca manusia).

In [ ]:
ringkasan_klaster = predictions.groupBy("prediction").avg("frekuensi_transaksi", "rata_rata_nilai") \
    .withColumnRenamed("avg(frekuensi_transaksi)", "rata2_frekuensi") \
    .withColumnRenamed("avg(rata_rata_nilai)", "rata2_nilai_transaksi") \
    .orderBy("prediction")

ringkasan_klaster.show()

Berdasarkan hasil di atas, kita dapat memberi **nama bisnis** pada tiap klaster, misalnya:
- **Klaster dengan frekuensi tinggi, nilai rendah** → "Pelanggan Setia Hemat" (sering belanja kebutuhan kecil)
- **Klaster dengan frekuensi rendah, nilai tinggi** → "Pembeli Sesekali Bernilai Tinggi" (jarang belanja, namun nominal besar)
- **Klaster moderat** → "Pelanggan Reguler"

Segmentasi seperti ini sangat berguna untuk tim marketing — misalnya, strategi promosi diskon kecil-kecilan cocok untuk kelompok pertama, sementara program loyalitas eksklusif lebih cocok untuk kelompok kedua.

### Visualisasi Klaster

In [ ]:
import pandas as pd

pdf_plot = predictions.select("frekuensi_transaksi", "rata_rata_nilai", "prediction").toPandas()

plt.figure(figsize=(8, 6))
warna = ["#1F3864", "#D9A441", "#1E7145"]
for klaster in sorted(pdf_plot["prediction"].unique()):
    subset = pdf_plot[pdf_plot["prediction"] == klaster]
    plt.scatter(subset["frekuensi_transaksi"], subset["rata_rata_nilai"],
                label=f"Klaster {klaster}", color=warna[klaster], alpha=0.6)

plt.title("Segmentasi Pelanggan Berdasarkan Perilaku Transaksi")
plt.xlabel("Frekuensi Transaksi per Bulan")
plt.ylabel("Rata-rata Nilai Transaksi (Rp)")
plt.legend()
plt.tight_layout()
plt.show()

> **Catatan teknis:** `toPandas()` menarik **seluruh** data dari Spark DataFrame ke memori lokal sebagai pandas DataFrame — aman untuk data hasil agregasi/ringkasan berukuran kecil seperti kebutuhan visualisasi ini (ingat Pertemuan 2), namun **jangan** dilakukan pada dataset mentah berskala jutaan/miliaran baris, karena akan menghilangkan seluruh manfaat pemrosesan terdistribusi Spark yang telah kita pelajari sepanjang semester.

---

## 14.6 Menyimpan Model (Reinforcement dari Pertemuan 13)

In [ ]:
path_model_kmeans = "model_segmentasi_pelanggan"

# Menyimpan KEDUA komponen: model scaler dan model K-Means, agar dapat dipakai ulang secara utuh
scaler_model.write().overwrite().save(f"{path_model_kmeans}_scaler")
model.write().overwrite().save(f"{path_model_kmeans}_kmeans")

print("Model scaler & K-Means berhasil disimpan.")
print("Untuk memuatnya kembali nanti, gunakan StandardScalerModel.load() dan KMeansModel.load().")

spark.stop()
print("SparkSession ditutup.")

---
## Latihan Mandiri

Jalankan ulang cell pembuatan `SparkSession` dan dataset (Sub-bab 14.2) sebelum mengerjakan latihan berikut.

In [ ]:
# Persiapan ulang untuk latihan
spark = SparkSession.builder.appName("Latihan14").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

df = spark.createDataFrame(data)
print("Siap untuk latihan. Total baris:", df.count())

**Soal 1.** Ulangi tahap `VectorAssembler` dan `StandardScaler` seperti Sub-bab 14.3, namun kali ini **tanpa** melakukan scaling (langsung gunakan `features_mentah` sebagai input `KMeans`, lewati `StandardScaler`). Latih model dengan `k=3`, lalu bandingkan `groupBy("prediction").count()`-nya dengan hasil Sub-bab 14.5 yang menggunakan scaling. Apakah hasilnya berbeda?

In [ ]:
# Jawaban Soal 1 di sini


**Soal 2.** Latih model K-Means dengan `k=2` (bukan 3) pada data yang sudah di-scaling. Bandingkan **Silhouette Score**-nya dengan model `k=3` pada Sub-bab 14.5 — apakah k=2 menghasilkan skor lebih baik atau lebih buruk?

In [ ]:
# Jawaban Soal 2 di sini


**Soal 3.** Menggunakan model `k=3` (Sub-bab 14.5), tampilkan `pelanggan_id` untuk **5 pelanggan** yang termasuk klaster dengan `rata2_nilai_transaksi` **tertinggi** (petunjuk: gabungkan hasil `ringkasan_klaster` untuk mengetahui klaster mana yang dimaksud, lalu `filter()` pada `predictions`).

In [ ]:
# Jawaban Soal 3 di sini


**Soal 4 (Refleksi singkat).** Dalam 2-3 kalimat: mengapa kita tidak bisa menggunakan metrik seperti *accuracy* atau *F1-score* (Pertemuan 12) untuk mengevaluasi hasil clustering K-Means? Tulis jawaban pada markdown cell di bawah ini.

*(Tulis jawaban di sini — jangan lupa `spark.stop()` setelah selesai berlatih)*


---
## TUGAS MANDIRI (Dikerjakan Selama 1 Minggu)

> **Tenggat waktu:** dikumpulkan paling lambat **sebelum Pertemuan 15 dimulai**.
> **Sifat tugas:** individu.

### Konteks / Skenario

"Kopi Nusantara" (skenario yang menemani kita sejak Pertemuan 7) ingin melakukan segmentasi terhadap **menu-menu** di katalog mereka (bukan pelanggan) — untuk membantu tim dapur memahami menu mana yang mirip karakteristiknya dari sisi harga dan popularitas, guna menyusun strategi bundling paket promo.

### Menyiapkan Dataset

Jalankan cell berikut untuk membuat dataset.

In [ ]:
import numpy as np
import pandas as pd

np.random.seed(140)
n_menu = 60

# Tiga pola menu: (1) murah & sangat populer, (2) mahal & jarang dipesan, (3) sedang & cukup populer
pola_1 = pd.DataFrame({
    "harga": np.random.normal(18000, 2000, 20),
    "rata_rata_pesanan_harian": np.random.normal(45, 8, 20),
})
pola_2 = pd.DataFrame({
    "harga": np.random.normal(45000, 5000, 20),
    "rata_rata_pesanan_harian": np.random.normal(8, 3, 20),
})
pola_3 = pd.DataFrame({
    "harga": np.random.normal(28000, 3000, 20),
    "rata_rata_pesanan_harian": np.random.normal(22, 5, 20),
})

data_tugas14 = pd.concat([pola_1, pola_2, pola_3], ignore_index=True)
data_tugas14["harga"] = data_tugas14["harga"].clip(lower=10000).round(-2)
data_tugas14["rata_rata_pesanan_harian"] = data_tugas14["rata_rata_pesanan_harian"].clip(lower=1).round(1)
data_tugas14["menu_id"] = [f"MENU-{i:03d}" for i in range(len(data_tugas14))]
data_tugas14 = data_tugas14.sample(frac=1, random_state=1).reset_index(drop=True)  # acak urutan barisnya

data_tugas14.to_csv("tugas14_menu.csv", index=False)
print(f"Dataset berhasil dibuat: {len(data_tugas14)} menu")
print(data_tugas14.head())

### Instruksi Pengerjaan

Buat notebook baru **`Tugas14_[NPM]_[Nama Lengkap].ipynb`**, buat `SparkSession`, baca `tugas14_menu.csv`, lalu kerjakan bagian **A sampai F** berikut:

---

**A. Feature Engineering** *(bobot 15%)*

Gabungkan kolom `harga` dan `rata_rata_pesanan_harian` menjadi satu vector menggunakan `VectorAssembler`, lalu terapkan `StandardScaler`. Jelaskan singkat (1 kalimat markdown) mengapa scaling diperlukan di sini.

**B. Elbow Method** *(bobot 20%)*

Latih model K-Means untuk **k=2 hingga k=6**, catat WSSSE tiap k, dan buat visualisasi grafik elbow method menggunakan matplotlib. Tentukan k optimal berdasarkan grafik tersebut, sertakan alasan singkat.

**C. Melatih Model Final** *(bobot 20%)*

Latih model K-Means final menggunakan k hasil pilihan bagian B. Hitung dan tampilkan **Silhouette Score**-nya.

**D. Interpretasi Bisnis** *(bobot 20%)*

Tampilkan rata-rata `harga` dan `rata_rata_pesanan_harian` untuk **setiap klaster** (skala asli, bukan hasil scaling). Beri **nama bisnis** untuk masing-masing klaster (mis. "Menu Andalan Terjangkau") beserta alasan singkat berdasarkan karakteristik angkanya.

**E. Visualisasi** *(bobot 15%)*

Buat scatter plot `harga` vs `rata_rata_pesanan_harian`, diwarnai berdasarkan hasil klaster (ikuti pola Sub-bab 14.5).

**F. Rekomendasi Bundling** *(bobot 10%)*

Tulis pada markdown cell (**minimal 80 kata**): berdasarkan segmentasi yang ditemukan, usulkan **satu ide paket bundling promo** yang masuk akal (mis. menggabungkan menu dari klaster tertentu), disertai alasan bisnisnya.

---

### Ketentuan Pengumpulan

- Kumpulkan `Tugas14_[NPM]_[Nama Lengkap].ipynb` melalui asisten praktikum, paling lambat **1 minggu dari hari ini, pukul 23.59 WIB**.
- Seluruh cell wajib sudah dijalankan (**Run All**) sebelum dikumpulkan.

### Rubrik Penilaian

| Bagian | Kriteria | Bobot |
|---|---|---|
| A. Feature Engineering | Assembler & scaler diterapkan benar, penjelasan tepat | 15% |
| B. Elbow Method | WSSSE dihitung untuk seluruh rentang k, grafik & pemilihan k logis | 20% |
| C. Model Final | Model terlatih benar, Silhouette Score dihitung & ditampilkan | 20% |
| D. Interpretasi Bisnis | Nama & alasan tiap klaster masuk akal berdasarkan data | 20% |
| E. Visualisasi | Scatter plot jelas, benar mewarnai berdasarkan klaster | 15% |
| F. Rekomendasi Bundling | Ide bundling relevan & alasan bisnis logis | 10% |
